# Limpieza Generacion Media Total y por tipos

Objetivo final:

datetime | year | month | day | hour | eolica | solar_fv | nuclear | ciclo_combinado | ...

Una fila = una hora
Una columna = una tecnología
Valores = float
Sin duplicados
Sin problemas de timezone

In [ ]:
import pandas as pd
import numpy as np
import re

# =========================
# 1. CARGA DEL CSV
# =========================

df = pd.read_csv(
    "GeneracionMediaTipos2020_2024.csv",
    sep=";",
    parse_dates=["datetime"]
)

# =========================
# 2. LIMPIEZA BASICA
# =========================

# Eliminar espacios en columnas
df.columns = df.columns.str.strip()

# Eliminar filas totalmente vacías
df = df.dropna(how="all")

# =========================
# 3. LIMPIEZA DE VALUE
# =========================

# Asegurarnos de que value es string
#df["value"] = df["value"].astype(str)

# Eliminar posibles separadores de miles (por seguridad)
#df["value"] = df["value"].str.replace(".", "", regex=False)\
#                           .str.replace(",", ".", regex=False)

#df["value"] = pd.to_numeric(df["value"], errors="coerce")

#funcion para eliminar los puntos execepto el primero, que indica los decimales
def clean_mixed_number(x):
    if x is None:
        return x
    
    s = str(x).strip()
    
    # Si no tiene puntos o solo hay uno, no hace falta hacer nada
    if s.count(".") <= 1:
        return float(s)
    
    # Dividimos la cadena por los puntos
    parts = s.split(".")
    
    # Unimos todo menos la ultima parte (los deciamles son a partir del primer punto a la derecha)
    integer_part = "".join(parts[:-1])
    decimal_part = parts[-1]
    
    cleaned = integer_part + "." + decimal_part
    
    # Si no tiene punto
    return float(cleaned)


# Aplicar la funcion a la columna value
df["value"] = df["value"].apply(clean_mixed_number)

# Eliminar filas con valores invalidos (na)
df = df.dropna(subset=["value"])

# =========================
# 4. LIMPIEZA DE NOMBRES DE TECNOLOGÍA
# =========================

# Quitar el prefijo que tienen todos los nombres
df["technology"] = df["name"].str.replace(
    "Generación medida ", "", regex=False
)

# Normalizar nombres (snake_case, sin tildes)
def clean_name(text):
    text = text.lower()
    text = re.sub(r"[áàä]", "a", text)
    text = re.sub(r"[éèë]", "e", text)
    text = re.sub(r"[íìï]", "i", text)
    text = re.sub(r"[óòö]", "o", text)
    text = re.sub(r"[úùü]", "u", text)
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")

# 
df["technology"] = df["technology"].apply(clean_name)

# =========================
# 5. LIMPIEZA DATETIME
# ==========================

# Convertir a datetime si no lo esta ya
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)

# Opcional: convertir a hora local España sin timezone
df["datetime"] = df["datetime"].dt.tz_convert("Europe/Madrid")
df["datetime"] = df["datetime"].dt.tz_localize(None)

# =========================
# 6. ELIMINAR POSIBLES DUPLICADOS
# =========================

df = df.drop_duplicates(subset=["datetime", "technology"])

# =========================
# 7. PIVOT A FORMATO ANCHO
# =========================

# En vez de estar en filas, pasarlo a columnas (es decir, df mas ancho)

df_pivot = df.pivot(
    index="datetime",
    columns="technology",
    values="value"
)

# Rellenar posibles huecos con 0
df_pivot = df_pivot.fillna(0)

# =========================
# 8. AÑADIR VARIABLES TEMPORALES
# ==========================

# Anadir mas variables para los modelos y dividir la fecha

df_pivot = df_pivot.sort_index()

df_pivot["year"] = df_pivot.index.year
df_pivot["month"] = df_pivot.index.month
df_pivot["day"] = df_pivot.index.day
df_pivot["hour"] = df_pivot.index.hour
df_pivot["dayofweek"] = df_pivot.index.dayofweek
df_pivot["is_weekend"] = (df_pivot["dayofweek"] >= 5).astype(int)

# =========================
# 9. REORDENAR COLUMNAS (opcional)
# =========================

time_cols = ["year", "month", "day", "hour", "dayofweek", "is_weekend"]
other_cols = [c for c in df_pivot.columns if c not in time_cols]

df_pivot = df_pivot[time_cols + other_cols]

# =========================
# 10. RESET INDEX (Solo si queremos dejar la col DATETIME)
# =========================

df_pivot = df_pivot.reset_index()

# =========================
# RESULTADO FINAL
# =========================

print(df_pivot.head())
print(df_pivot.info())

df_pivot.to_csv("LimpizaGeneracionMediaTipos2020_2024V4.csv", index=False, sep=';')


technology            datetime  year  month  day  hour  dayofweek  is_weekend  \
0          2020-01-01 00:00:00  2020      1    1     0          2           0   
1          2020-01-01 01:00:00  2020      1    1     1          2           0   
2          2020-01-01 02:00:00  2020      1    1     2          2           0   
3          2020-01-01 03:00:00  2020      1    1     3          2           0   
4          2020-01-01 04:00:00  2020      1    1     4          2           0   

technology  biogas  biomasa  ciclo_combinado  ...   nuclear  \
0           89.980  273.283         4375.106  ...  7108.765   
1           90.457  269.422         4538.635  ...  7104.832   
2           85.455  261.432         4177.421  ...  7108.460   
3           85.664  258.927         3939.211  ...  7105.782   
4           85.909  259.170         4163.001  ...  7106.640   

technology  oceano_y_geotermica  residuos_domesticos_y_similares  \
0                         3.516                          179.551  

### Tal vez deberíamos eliminar la columna de Generacion Total

Esa NO deberías usarla como feature si estás usando todas las tecnologías, porque:

total = suma(tecnologías)

Sería data leakage parcial.

In [ ]:
if "total_tipo_produccion" in df_pivot.columns:
    df_pivot = df_pivot.drop(columns=["total_tipo_produccion"])

# Validacion de los datos

Antes de unir con demanda o precio, hay que verificar tres cosas:

### 1.  ¿Tenemos todas las horas?
Entre 2020 y 2024 deben haber:

24 horas × 365 días

ajustar años bisiestos

el ajuste horario ( el cambio de hora)

### 2. ¿Existen valores negativos?
No deberían existir en generación (salvo bombeo si estuviera modelado como consumo).

### 3. ¿Hay picos absurdos?
Ejemplo:

Nuclear unos 7000 MW constante

Solar FV puede llegar a miles en 2023-2024

Eólica variable pero no millones

In [ ]:
# 1 Comprobar que estan todas las horas
print("Número de filas:", len(df))
print("Fechas mín / máx:", df.index.min(), df.index.max())

expected = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="H"
)

missing = expected.difference(df.index)

print("Horas faltantes:", len(missing))


# 2 Comprobar valores negativos


# 3 Comprobar maximos y minimos sean normales


Número de filas: 791612
Fechas mín / máx: 0 791683
Horas faltantes: 1


C:\Users\espar\AppData\Local\Temp\ipykernel_11876\274627601.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  expected = pd.date_range(


,min,max
id,1150.0,10195.0
value,-686.661,43507.695
datetime,2020-01-01 00:00:00,2024-12-31 23:00:00
